# 00 — Prepare the revision dataset, freeze the grouped split, and cache I3D features

This notebook is designed specifically for the Scientific Reports major revision.

**Important:** it does **not** recreate the 29-frame dataset. It assumes the preprocessing from `attempt 5 - Final` has already been completed and that each retained stroke folder contains:

- `frame_0000.jpg` ... `frame_0028.jpg`
- `phase_scores.txt`
- `pose_body25_interp.npy`
- optionally `pose_body25_observed.npy`
- optionally `stroke_attributes.txt`

The notebook does four things once:

1. Rebuilds the ordered list of valid 29-frame strokes used in the revision experiments.
2. Freezes the source-video grouped train/validation/test split to disk.
3. Caches the cleaned pose tensors and pose-quality information.
4. Extracts and saves I3D features for both the 29-frame representation and a deterministic 12-frame representation derived from the same 29-frame strokes.

After this notebook has completed successfully, **do not rerun the old dataset-construction cells** for the revision experiments. The later notebooks load these cached arrays directly.

### Controlled design

- 29 frames: 14 buildup / 7 execution / 8 follow-through.
- 12 frames: 4 buildup / 3 execution / 5 follow-through.
- The 12 frames are deterministically selected from the canonical 29-frame sample.
- All conditions use the same strokes, labels, source videos, and grouped split.

This gives us a clean temporal comparison (RGB-12 vs RGB-29) and clean modality comparison (RGB-29 vs Pose-29 vs RGB+Pose-29).

In [ ]:
from pathlib import Path
import os, re, json, sys
import numpy as np
import pandas as pd
import cv2

# ============================================================
# CONFIG
# ============================================================
# Required: parent folder containing processed_strokes_*_interp_29 directories.
data_root_env = os.environ.get("AQA_DATA_ROOT")
if not data_root_env:
    raise EnvironmentError(
        "Set AQA_DATA_ROOT to the folder containing processed_strokes_*_interp_29 directories."
    )
DATA_ROOT = Path(data_root_env).expanduser().resolve()

# Optional: output/cache location. Defaults to ./artifacts.
REVISION_ROOT = Path(
    os.environ.get("AQA_REVISION_ROOT", str(Path.cwd() / "artifacts"))
).expanduser().resolve()

# Optional: folder containing i3d_inception.py if it is not already importable.
i3d_code_env = os.environ.get("AQA_I3D_CODE_DIR")
I3D_CODE_DIR = Path(i3d_code_env).expanduser().resolve() if i3d_code_env else None

VIDEO_PATTERN = "processed_strokes_*_interp_29"
N_FRAMES_29 = 29
IMG_SIZE = (128, 128)
SEED = 42

# Set True only if you deliberately want to regenerate cached I3D features.
FORCE_REBUILD_FEATURES = False

REVISION_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = REVISION_ROOT / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT    :", DATA_ROOT)
print("REVISION_ROOT:", REVISION_ROOT)
print("I3D_CODE_DIR :", I3D_CODE_DIR)
assert DATA_ROOT.exists(), f"DATA_ROOT does not exist: {DATA_ROOT}"


In [ ]:
# ============================================================
# Helpers: labels, metadata, pose interpolation
# ============================================================
SCORE_RE = re.compile(
    r".*\((Buildup|Execution|FollowThrough)\):\s*"
    r"Head=([-+]?\d*\.?\d+),\s*"
    r"Shoulder=([-+]?\d*\.?\d+),\s*"
    r"Hands=([-+]?\d*\.?\d+),\s*"
    r"Hips=([-+]?\d*\.?\d+),\s*"
    r"Feet=([-+]?\d*\.?\d+)"
)

def load_phase_scores(scores_file):
    arr = []
    with open(scores_file, "r", encoding="utf-8") as f:
        for line in f:
            m = SCORE_RE.match(line.strip())
            if m:
                arr.append([float(m.group(i)) for i in range(2, 7)])
    return np.asarray(arr, dtype=np.float32)

def parse_attributes(path):
    out = {
        "Handedness": "N/A",
        "Foot_Type": "N/A",
        "Stroke_Type": "N/A",
        "Dismissal": "N/A",
    }
    if not path.exists():
        return out
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if ":" not in line:
                continue
            k, v = line.split(":", 1)
            k, v = k.strip(), v.strip()
            if k in out:
                out[k] = v
    return out

def extract_video_id(path_obj):
    for part in Path(path_obj).parts:
        if part.startswith("processed_strokes_") and part.endswith("_interp_29"):
            return part.replace("processed_strokes_", "").replace("_interp_29", "")
    return "UNKNOWN"

def interp_1d_nan(x):
    x = np.asarray(x, dtype=np.float32)
    idx = np.arange(len(x))
    good = np.isfinite(x)
    if good.sum() == 0:
        return np.zeros_like(x, dtype=np.float32)
    if good.sum() == 1:
        return np.full_like(x, x[good][0], dtype=np.float32)
    return np.interp(idx, idx[good], x[good]).astype(np.float32)

def clean_pose(pose):
    """Match Attempt 5: interpolate x/y over time, preserve confidence."""
    out = np.asarray(pose, dtype=np.float32).copy()
    out[..., :2] = np.clip(out[..., :2], 0, 1)
    for j in range(out.shape[1]):
        for d in range(2):
            out[:, j, d] = interp_1d_nan(out[:, j, d])
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def evenly_spaced_indices(start, stop_exclusive, target_n):
    local = np.linspace(0, (stop_exclusive - start) - 1, target_n)
    local = np.round(local).astype(int)
    return (local + start).tolist()

# 4 buildup, 3 execution, 5 follow-through selected from 14/7/8.
IDX_12 = np.asarray(
    evenly_spaced_indices(0, 14, 4)
    + evenly_spaced_indices(14, 21, 3)
    + evenly_spaced_indices(21, 29, 5),
    dtype=int,
)
print("12-frame indices from canonical 29:", IDX_12.tolist())
assert len(IDX_12) == 12

In [ ]:
# ============================================================
# Build the canonical manifest using the SAME ordering as Attempt 5
# ============================================================
records = []
y_scores_list = []
pose_list = []
pose_conf_list = []
observed_list = []

video_dirs = sorted([
    d for d in DATA_ROOT.glob(VIDEO_PATTERN)
    if d.is_dir() and "_cropped" not in d.name
])

print("29-frame source-video folders found:", len(video_dirs))

for vdir in video_dirs:
    stroke_dirs = sorted([
        s for s in vdir.iterdir()
        if s.is_dir() and s.name.startswith("stroke_")
    ])

    for sdir in stroke_dirs:
        # Same eligibility logic as Attempt 5, with an explicit 29-frame check.
        pose_path = sdir / "pose_body25_interp.npy"
        scores_path = sdir / "phase_scores.txt"

        if not pose_path.exists() or not scores_path.exists():
            continue

        frame_paths = [sdir / f"frame_{t:04d}.jpg" for t in range(29)]
        if not all(p.exists() for p in frame_paths):
            continue

        try:
            pose_raw = np.load(pose_path)
        except Exception:
            continue

        if pose_raw.shape != (29, 25, 3):
            continue

        scores = load_phase_scores(scores_path)
        if scores.shape != (3, 5):
            continue

        attrs = parse_attributes(sdir / "stroke_attributes.txt")
        observed_path = sdir / "pose_body25_observed.npy"
        if observed_path.exists():
            observed = np.load(observed_path)
            if observed.shape != (29, 25):
                observed = (np.isfinite(pose_raw[..., 0]) & np.isfinite(pose_raw[..., 1])).astype(np.uint8)
        else:
            observed = (np.isfinite(pose_raw[..., 0]) & np.isfinite(pose_raw[..., 1])).astype(np.uint8)

        video_id = extract_video_id(sdir)
        sample_id = len(records)

        records.append({
            "sample_id": sample_id,
            "source_video_id": video_id,
            "stroke_path": str(sdir),
            "stroke_folder": sdir.name,
            **attrs,
        })

        # Preserve raw confidence before x/y interpolation for pose-quality analysis.
        pose_conf_list.append(np.nan_to_num(pose_raw[..., 2], nan=0.0).astype(np.float32))
        observed_list.append(observed.astype(np.uint8))
        pose_list.append(clean_pose(pose_raw))
        y_scores_list.append(scores.astype(np.float32))

manifest = pd.DataFrame(records)
y_scores = np.stack(y_scores_list).astype(np.float32)
X_pose = np.stack(pose_list).astype(np.float32)
X_pose_conf = np.stack(pose_conf_list).astype(np.float32)
X_pose_observed = np.stack(observed_list).astype(np.uint8)

print("Valid strokes:", len(manifest))
print("Unique source videos:", manifest["source_video_id"].nunique())
print("y_scores:", y_scores.shape)
print("X_pose:", X_pose.shape)

if len(manifest) != 8151:
    print(
        "WARNING: Attempt 5 reported 8,151 strokes. "
        f"This scan found {len(manifest)}. Do NOT train until the discrepancy is understood."
    )

manifest.to_csv(CACHE_DIR / "manifest.csv", index=False)
np.save(CACHE_DIR / "y_scores.npy", y_scores)
np.save(CACHE_DIR / "pose29_clean.npy", X_pose)
np.save(CACHE_DIR / "pose29_confidence.npy", X_pose_conf)
np.save(CACHE_DIR / "pose29_observed.npy", X_pose_observed)

In [ ]:
# ============================================================
# Reproduce Attempt 5's source-video grouped split and FREEZE it
# ============================================================
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.75, 0.15, 0.10
rng = np.random.RandomState(SEED)

groups = manifest["source_video_id"].astype(str).to_numpy()
unique_videos = np.asarray(sorted(np.unique(groups)))
n_videos = len(unique_videos)

perm = rng.permutation(n_videos)
n_train_v = int(np.floor(TRAIN_RATIO * n_videos))
n_val_v = int(np.floor(VAL_RATIO * n_videos))
n_test_v = n_videos - n_train_v - n_val_v

if n_videos >= 3:
    n_val_v = max(1, n_val_v)
    n_test_v = max(1, n_test_v)
    n_train_v = n_videos - n_val_v - n_test_v

train_videos = set(unique_videos[perm[:n_train_v]])
val_videos = set(unique_videos[perm[n_train_v:n_train_v + n_val_v]])
test_videos = set(unique_videos[perm[n_train_v + n_val_v:]])

train_idx = np.where(np.isin(groups, list(train_videos)))[0]
val_idx = np.where(np.isin(groups, list(val_videos)))[0]
test_idx = np.where(np.isin(groups, list(test_videos)))[0]

assert set(train_idx).isdisjoint(set(val_idx))
assert set(train_idx).isdisjoint(set(test_idx))
assert set(val_idx).isdisjoint(set(test_idx))
assert len(train_idx) + len(val_idx) + len(test_idx) == len(manifest)

split = np.full(len(manifest), "", dtype=object)
split[train_idx] = "train"
split[val_idx] = "validation"
split[test_idx] = "test"

manifest["split"] = split
manifest.to_csv(CACHE_DIR / "manifest_with_split.csv", index=False)
np.save(CACHE_DIR / "train_idx.npy", train_idx)
np.save(CACHE_DIR / "val_idx.npy", val_idx)
np.save(CACHE_DIR / "test_idx.npy", test_idx)

print("Video split  :", len(train_videos), len(val_videos), len(test_videos))
print("Sample split :", len(train_idx), len(val_idx), len(test_idx))

EXPECTED = (5806, 1378, 967)
actual = (len(train_idx), len(val_idx), len(test_idx))
if actual != EXPECTED:
    print(f"WARNING: expected Attempt 5 sample split {EXPECTED}, obtained {actual}.")
else:
    print("SUCCESS: grouped sample split exactly matches Attempt 5.")

## I3D feature caching

This is the expensive part, but it should be done only once.

Unlike Attempt 5, the notebook does **not** load all 8,151 RGB sequences into one huge float32 array. Frames are streamed from disk, converted to four-frame I3D windows, and the resulting feature tensors are written directly to `.npy` files.

Both the 29-frame and 12-frame feature tensors are extracted in the same pass through the image folders.

In [ ]:
# ============================================================
# Load the I3D backbone used in Attempt 5
# ============================================================
if I3D_CODE_DIR is not None:
    I3D_CODE_DIR = Path(I3D_CODE_DIR)
    if str(I3D_CODE_DIR) not in sys.path:
        sys.path.insert(0, str(I3D_CODE_DIR))

try:
    from i3d_inception import Inception_Inflated3d
except Exception as e:
    raise ImportError(
        "Could not import i3d_inception. Run this notebook in the same environment/folder "
        "as Attempt 5, or set I3D_CODE_DIR above to the folder containing i3d_inception.py."
    ) from e

i3d_model = Inception_Inflated3d(
    include_top=False,
    weights="rgb_imagenet_and_kinetics",
    input_shape=(4, 128, 128, 3)
)
i3d_model.trainable = False

feat_dim = int(i3d_model.output_shape[-1])
print("I3D output feature dimension:", feat_dim)

In [ ]:
# ============================================================
# Streaming I3D extraction for BOTH 29-frame and 12-frame inputs
# ============================================================
from numpy.lib.format import open_memmap
from tqdm import tqdm

RAW29 = CACHE_DIR / "rgb29_i3d_raw.npy"
RAW12 = CACHE_DIR / "rgb12_i3d_raw.npy"

def load_frames_29(stroke_path):
    sdir = Path(stroke_path)
    frames = []
    for t in range(29):
        img = cv2.imread(str(sdir / f"frame_{t:04d}.jpg"))
        if img is None:
            raise RuntimeError(f"Could not read {sdir / f'frame_{t:04d}.jpg'}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        frames.append(img.astype(np.float32) / 255.0)
    return np.stack(frames, axis=0)

def make_window(frames, t, window=4):
    T, H, W, C = frames.shape
    start = max(0, t - window // 2)
    end = min(T, start + window)
    if end - start < window:
        start = max(0, end - window)
    clip = frames[start:end]
    if len(clip) < window:
        pad = np.zeros((window - len(clip), H, W, C), dtype=clip.dtype)
        clip = np.concatenate([clip, pad], axis=0)
    return clip

def flush_batch(clips, locations, out29, out12):
    if not clips:
        return
    batch = np.stack(clips, axis=0)
    pred = i3d_model.predict(batch, verbose=0)
    # Attempt 5 used pred[j, 0, 0, 0, :]
    pred = pred[:, 0, 0, 0, :].astype(np.float32)

    for j, (which, sample_i, time_i) in enumerate(locations):
        if which == "29":
            out29[sample_i, time_i] = pred[j]
        else:
            out12[sample_i, time_i] = pred[j]

def extract_both(batch_size=64):
    n = len(manifest)
    out29 = open_memmap(RAW29, mode="w+", dtype=np.float32, shape=(n, 29, feat_dim))
    out12 = open_memmap(RAW12, mode="w+", dtype=np.float32, shape=(n, 12, feat_dim))

    clips, locations = [], []

    for i, row in tqdm(manifest.iterrows(), total=n, desc="Caching I3D 29 + 12"):
        f29 = load_frames_29(row["stroke_path"])
        f12 = f29[IDX_12]

        for which, frames in [("29", f29), ("12", f12)]:
            for t in range(len(frames)):
                clips.append(make_window(frames, t, window=4))
                locations.append((which, i, t))

                if len(clips) >= batch_size:
                    flush_batch(clips, locations, out29, out12)
                    clips, locations = [], []

    flush_batch(clips, locations, out29, out12)
    out29.flush()
    out12.flush()

if FORCE_REBUILD_FEATURES or not (RAW29.exists() and RAW12.exists()):
    extract_both(batch_size=64)
    print("Raw feature caches written.")
else:
    print("Raw I3D feature caches already exist; extraction skipped.")
    print("Set FORCE_REBUILD_FEATURES=True only if you intentionally want to rebuild them.")

In [ ]:
# ============================================================
# Train-only feature normalisation, cached once
# ============================================================
NORM29 = CACHE_DIR / "rgb29_i3d_norm.npy"
NORM12 = CACHE_DIR / "rgb12_i3d_norm.npy"
STATS = CACHE_DIR / "rgb_feature_normalisation_stats.npz"

def compute_train_stats(arr, train_indices, chunk=256):
    feat_dim_local = arr.shape[-1]
    s = np.zeros(feat_dim_local, dtype=np.float64)
    ss = np.zeros(feat_dim_local, dtype=np.float64)
    count = 0

    for start in range(0, len(train_indices), chunk):
        idx = train_indices[start:start+chunk]
        x = np.asarray(arr[idx], dtype=np.float32)
        s += x.sum(axis=(0, 1), dtype=np.float64)
        ss += np.square(x, dtype=np.float64).sum(axis=(0, 1), dtype=np.float64)
        count += x.shape[0] * x.shape[1]

    mean = s / count
    var = np.maximum(ss / count - mean**2, 0.0)
    std = np.sqrt(var) + 1e-6
    return mean.astype(np.float32), std.astype(np.float32)

def normalise_to_file(raw_path, out_path, train_indices, chunk=256):
    raw = np.load(raw_path, mmap_mode="r")
    mean, std = compute_train_stats(raw, train_indices, chunk=chunk)

    out = open_memmap(out_path, mode="w+", dtype=np.float32, shape=raw.shape)
    for start in tqdm(range(0, raw.shape[0], chunk), desc=f"Normalising {raw_path.name}"):
        x = np.asarray(raw[start:start+chunk], dtype=np.float32)
        out[start:start+len(x)] = (x - mean[None, None, :]) / std[None, None, :]
    out.flush()
    return mean, std

if FORCE_REBUILD_FEATURES or not (NORM29.exists() and NORM12.exists() and STATS.exists()):
    mean29, std29 = normalise_to_file(RAW29, NORM29, train_idx)
    mean12, std12 = normalise_to_file(RAW12, NORM12, train_idx)
    np.savez(STATS, mean29=mean29, std29=std29, mean12=mean12, std12=std12)
    print("Normalised feature caches written.")
else:
    print("Normalised feature caches already exist; normalisation skipped.")

In [ ]:
# ============================================================
# Final cache verification and summary
# ============================================================
summary = {
    "n_samples": int(len(manifest)),
    "n_source_videos": int(manifest["source_video_id"].nunique()),
    "train_samples": int(len(train_idx)),
    "validation_samples": int(len(val_idx)),
    "test_samples": int(len(test_idx)),
    "train_videos": int(len(train_videos)),
    "validation_videos": int(len(val_videos)),
    "test_videos": int(len(test_videos)),
    "frames_29": [14, 7, 8],
    "frames_12": [4, 3, 5],
    "idx12_from_29": IDX_12.tolist(),
    "split_seed": int(SEED),
    "note": "Split guarantees source-video independence, not player independence."
}

with open(CACHE_DIR / "prep_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

required = [
    "manifest_with_split.csv", "y_scores.npy", "pose29_clean.npy",
    "pose29_confidence.npy", "pose29_observed.npy",
    "train_idx.npy", "val_idx.npy", "test_idx.npy",
    "rgb29_i3d_norm.npy", "rgb12_i3d_norm.npy",
]
missing = [x for x in required if not (CACHE_DIR / x).exists()]
assert not missing, f"Missing cache files: {missing}"

print("\nPREPARATION COMPLETE.")
print("You can now run notebooks 01–04 without recreating the dataset or re-extracting I3D features.")